# Goals

Determine how the bug described in https://github.com/IQSS/dataverse.harvard.edu/issues/500 can adversley affect search results of users of Harvard Dataverse and other Dataverse repositories.

# Plan

- Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

- For Harvard Dataverse:
    - Get the list of URLs and names of RORs in the externalvocabularyvalue table

    - Get the list of author affiliations with RORs; the list and count of datasets that have those author affiliations; and if each ROR is in the externalvocabularyvalue table
    - For each author affiliation with a ROR:
        - Search 1: Use the Search API to search for datasets by the name of the author affiliation
        - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
        - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

- For other Dataverse installations:
    - Determine which Dataverse installations have datasets with ROR URLs in their Author Affiliations or Funding Information Agencies
    - For each installation:

        - Get the list of author affiliations with RORs and the list and count of datasets that have those author affiliations (I don't know what's in each installation's externalvocabularyvalue table, so I can't include if each ROR is in the externalvocabularyvalue table)
        - For each author affiliation with a ROR:
            - Search 1: Use the Search API to search for datasets by the name of the author affiliation
            - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
            - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

# Import libraries and set variables

In [ ]:
import csv
import json
import os
import platform
import re
import sys
from pathlib import Path
from time import sleep

import pandas as pd
import requests
from tqdm import tqdm

tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions, dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

rorRegex = r'https?://ror\.org/0[a-hj-km-np-tv-z|0-9]{6}[0-9]{2}$'

# Collect affiliation names, RORs and other info

## Parse Dataverse JSON exports to get author affiliation metadata

In [ ]:
# Get the Affiliation Names and ROR URLs that depositors have entered in the Author Affiliation 
# fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

collectionRange = '2026.08.25-2026.09.02'

authorAffiliationCsvFilePath = os.path.join(
    desktopPath, f'author_affiliations_{collectionRange}.csv')

# Create CSV file and add headerrow
with open(authorAffiliationCsvFilePath, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'dataset_pid', 'latest_dataset_version_number',
        'expanded_value_key_present',
        'author_affiliation_name', 'author_affiliation_ror'])

dataverseJsonExportsDirectory = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/all_json_metadata_files'
jsonFilePaths = list(Path(dataverseJsonExportsDirectory).glob('*(latest_version).json'))

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=jsonFilePaths)
for jsonFilePath in loopObj:
    jsonFileName = os.path.basename(jsonFilePath)
    loopObj.set_postfix_str(jsonFileName)
    with open(jsonFilePath, 'r', encoding='utf-8') as f:
        datasetMetadata = json.load(f)

        # Get names of metadata blocks and name of first metadata block whose name matches the give metadatablockName
        matchingMetadataBlockName = None
        metadataBlockNamesList = []
        metadatablockDict = common_functions.improved_get(datasetMetadata, 'data.metadataBlocks')
        for metadatablock in metadatablockDict:
            metadataBlockNamesList.append(metadatablock)

        metadatablockName = 'citation'
        if metadatablockName in metadataBlockNamesList:
            matchingMetadataBlockName = metadatablockName

        # If the exact metadata block name is not in the list, do a less exact search
        elif metadatablockName not in metadataBlockNamesList:
            matchingMetadataBlockNames = [s for idx, s in enumerate(metadataBlockNamesList) if metadatablockName in s]
            if matchingMetadataBlockNames:
                matchingMetadataBlockName = matchingMetadataBlockNames[0]

        if matchingMetadataBlockName is not None:
            citationMetadataBlockDict = datasetMetadata['data']['metadataBlocks'][matchingMetadataBlockName]
            for field in citationMetadataBlockDict['fields']:
                if field['typeName'] == 'author':
                    authorsDict = field
                    for author in authorsDict['value']:
                        if 'authorAffiliation' in author:

                            # Get basic metadata
                            basicInfoDict = datasetMetadata['data']
                            keysToRemove = ['metadataBlocks', 'files']
                            basicInfoDict = {key: value for key, value in basicInfoDict.items() if key not in keysToRemove}

                            datasetPid = basicInfoDict['datasetPersistentId']

                            majorVersionNumber = basicInfoDict['versionNumber']
                            minorVersionNumber = basicInfoDict['versionMinorNumber']
                            datasetVersionNumber = f'{majorVersionNumber}.{minorVersionNumber}'

                            # Get Author affiliation metadata
                            authorAffiliationDict = author['authorAffiliation']
                            authorAffiliationValue = authorAffiliationDict['value']

                            if 'expandedvalue' in authorAffiliationDict:
                                expandedValueKeyPresent = True
                                authorAffiliationExpandedValue = authorAffiliationDict['expandedvalue']
                                authorAffiliationName = common_functions.improved_get(authorAffiliationExpandedValue, 'termName')
                                # authorAffiliationName = authorAffiliationExpandedValue['termName']
                                
                                # Use rorRegex to verify authorAffiliationValue is a valid ROR URL
                                if re.match(rorRegex, authorAffiliationValue):
                                    authorAffiliationRor = authorAffiliationValue

                            elif 'expandedvalue' not in authorAffiliationDict:
                                expandedValueKeyPresent = False
                                authorAffiliationValue = authorAffiliationValue
                                # Use rorRegex to verify authorAffiliationValue is a valid ROR URL
                                if re.match(rorRegex, authorAffiliationValue):
                                    authorAffiliationName = None
                                    authorAffiliationRor = authorAffiliationValue
                                else:
                                    authorAffiliationName = authorAffiliationValue
                                    authorAffiliationRor = None

                            affiliationToAdd = [
                                datasetPid, datasetVersionNumber, expandedValueKeyPresent,
                                authorAffiliationName, authorAffiliationRor
                            ]

                            # Write values to CSV file
                            with open(authorAffiliationCsvFilePath, mode='a', newline='', encoding='utf-8-sig') as f:
                                writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                                writer.writerow(affiliationToAdd)
                            

## Reformat author affiliation info

In [ ]:
authorAffiliationCsvFilePath = os.path.join(
    'data', 'processed', 
    'author_affiliations_2026.08.25-2026.09.02.csv')

authorAffiliationsDf = pd.read_csv(
    authorAffiliationCsvFilePath,
    encoding='utf-8-sig')

print(f'Rows in authorAffiliationsDf: {len(authorAffiliationsDf)}')

distinctDatasetsCount = len(list(set(authorAffiliationsDf['dataset_pid'].to_list())))
print(f'Count of distinct dataset PIDs in authorAffiliationsDf: {distinctDatasetsCount}')

authorAffiliationsDf.head()

In [ ]:
# To author_affiliations_2026.08.25-2026.09.02.csv, add columns to show the installation name and installation URL of each dataset.
# Use /Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataset_pids_from_most_known_dataverse_installations.csv

allInstallationMetadata2026Directory = os.path.join(
        '/Users/juliangautier/Documents',
        'all_installation_metadata_2026.08.25_22.57.59')

datasetsByInstallationsDf = pd.read_csv(
    os.path.join(
        allInstallationMetadata2026Directory,
        'dataset_pids_from_most_known_dataverse_installations.csv'),
    usecols=[
        'dataverse_installation_name',
        'dataverse_installation_url',
        'dataset_pid', 'dataset_pid_url'],
    dtype=str,
    encoding='utf-8-sig')
datasetsByInstallationsDf.head()

print(f'Rows in datasetsByInstallationsDf: {len(datasetsByInstallationsDf)}')

distinctInstallationsCount = len(list(set(datasetsByInstallationsDf['dataverse_installation_name'].to_list())))
print(f'Count of distinct installations in datasetsByInstallationsDf: {distinctInstallationsCount}')

datasetsByInstallationsDf.head()

In [ ]:
# Merge the two dataframes
authorAffiliationsDf2 = authorAffiliationsDf.merge(
    datasetsByInstallationsDf, how='left',
    on=['dataset_pid'])

authorAffiliationsDf2 = authorAffiliationsDf2[[
    'dataverse_installation_name', 'dataverse_installation_url',
    'dataset_pid', 'dataset_pid_url', 'latest_dataset_version_number',
    'expanded_value_key_present', 'author_affiliation_name', 'author_affiliation_ror'
]]

print(f'Rows in authorAffiliationsDf2: {len(authorAffiliationsDf2)}')

distinctInstallationsCount = len(list(set(authorAffiliationsDf2['dataverse_installation_name'].to_list())))
distinctDatasetsCount = len(list(set(authorAffiliationsDf2['dataset_pid'].to_list())))
print(f'Count of distinct installations in authorAffiliationsDf2: {distinctInstallationsCount}')
print(f'Count of distinct dataset PIDs in authorAffiliationsDf2: {distinctDatasetsCount}')

authorAffiliationsDf2.to_csv(
    os.path.join('data', 'processed', 'authorAffiliationsDf2.csv'), 
    index=False, encoding='utf-8-sig')

authorAffiliationsDf2.head()

## Data quality check of authorAffiliationsDf2

In [ ]:
# Data quality check: find dataset_pid values that appear more than once in datasetsByInstallationsDf
# and list the installation(s) each one is in

# keep=False marks every row of a duplicated dataset_pid, not just the later ones

duplicatePidRowsDf = authorAffiliationsDf2[
    authorAffiliationsDf2.duplicated(subset=['dataset_pid'], keep=False)]

# One row per duplicated dataset_pid, with the row count and the distinct installations it's in
duplicatePidsDf = (
    duplicatePidRowsDf
    .groupby('dataset_pid', as_index=False)
    .agg(
        row_count=('dataset_pid', 'size'),
        installation_count=('dataverse_installation_name', 'nunique'),
        installation_names=('dataverse_installation_name', lambda x: sorted(set(x.dropna()))),
        installation_urls=('dataverse_installation_url', lambda x: sorted(set(x.dropna()))))
    .sort_values('row_count', ascending=False))

print(f'Duplicated dataset_pid values in more than one installation: {(duplicatePidsDf["installation_count"] > 1).sum()}')

duplicatePidsDf.to_csv(
    os.path.join('data', 'processed', 'duplicate_dataset_pids_by_installation.csv'),
    index=False, encoding='utf-8-sig')

## Get list of unique ROR URLs and their organization names
From author_affiliations_2026.08.25-2026.09.02.csv, get the unique ROR URLs and names of organizations that depositors have entered in the Author Affiliation fields.

In [ ]:
authorAffiliationCsvFilePath = os.path.join(
    'data', 'processed', 
    'author_affiliations_2026.08.25-2026.09.02.csv')

authorAffiliationRorsDf = pd.read_csv(
    authorAffiliationCsvFilePath,
    usecols=['author_affiliation_name', 'author_affiliation_ror'],
    dtype=str,
    encoding='utf-8-sig')

authorAffiliationRorsDf.head()

In [ ]:
# Data quality checks: missing values, duplicates, values in author_affiliation_ror column 
# that aren't valid ROR URLs

print(f'Rows: {len(authorAffiliationRorsDf)}')
print(f'Rows with no ROR URL: {authorAffiliationRorsDf["author_affiliation_ror"].isna().sum()}')
print(f'Rows with a ROR URL but no organization name: {(authorAffiliationRorsDf["author_affiliation_ror"].notna() & authorAffiliationRorsDf["author_affiliation_name"].isna()).sum()}')
print(f'Fully duplicated rows: {authorAffiliationRorsDf.duplicated().sum()}')

rorUrlsList = list(authorAffiliationRorsDf['author_affiliation_ror'].dropna())
rorUrlsList = list(set(rorUrlsList))
print(f'Distinct ROR URLs in rorUrlsList: {len(rorUrlsList)}')

nonMatchingRorUrls = [i for i in rorUrlsList if not re.match(rorRegex, i)]
print(f'Malformed ROR URLs: {len(nonMatchingRorUrls)}')

In [ ]:
# Create a dataframe with one row per distinct ROR URL, a list of each distinct organization name for that ROR URL
# and the count of organization names. Missing names are skipped, so a ROR URL with no name in any row gets an empty list
authorAffiliationRorsDf = (
    authorAffiliationRorsDf
    .dropna(subset=['author_affiliation_ror'])
    .groupby('author_affiliation_ror', as_index=False)
    .agg(organization_names_from_expandedvalue_key=('author_affiliation_name', lambda x: sorted(set(x.dropna()))))
    .rename(columns={'author_affiliation_ror': 'ror_url'}))

authorAffiliationRorsDf['organization_name_count_from_expandedvalue_key'] = authorAffiliationRorsDf['organization_names_from_expandedvalue_key'].apply(len)
authorAffiliationRorsDf.head()

In [ ]:
# Data quality checks: missing values, duplicates, values in author_affiliation_ror column 
# that aren't valid ROR URLs

print(f'Rows: {len(authorAffiliationRorsDf)}')
rorUrlsList = list(authorAffiliationRorsDf['ror_url'].dropna())
rorUrlsList = list(set(rorUrlsList))
print(f'Distinct ROR URLs in rorUrlsList: {len(rorUrlsList)}')

print(f'Rows with no ROR URL: {authorAffiliationRorsDf["ror_url"].isna().sum()}')
print(f'Rows with a ROR URL but no organization name: {(authorAffiliationRorsDf["organization_name_count_from_expandedvalue_key"] == 0).sum()}')

nonMatchingRorUrls = [i for i in rorUrlsList if not re.match(rorRegex, i)]
print(f'Malformed ROR URLs: {len(nonMatchingRorUrls)}')

# authorAffiliationRorsDf.to_csv(
#     os.path.join('data', 'processed', 'ror_urls_and_organization_names.csv'), 
#     index=False, encoding='utf-8-sig')

In [ ]:
# Add a column, 'display_names_2026.11.01', to authorAffiliationRorsWithDisplayNamesDf, 
# that shows the display names of each ROR URL in the ror_url column

displayNamesColumnName = 'ror_display_names_2026_11_01'

authorAffiliationRorsWithDisplayNamesDf = authorAffiliationRorsDf

# Get the display name of each distinct ROR URL from the ROR API (one request per ROR URL)
rorUrlsList = list(authorAffiliationRorsWithDisplayNamesDf['ror_url'].dropna().unique())
displayNamesDict = {}

loopObj = tqdm(bar_format=tqdmBarFormat, iterable=rorUrlsList)
for rorUrl in loopObj:
    loopObj.set_postfix_str(rorUrl)
    rorInfoDict = common_functions.get_org_data_from_ror(rorUrl)
    displayNamesDict[rorUrl] = rorInfoDict['displayName']
    sleep(.25)

authorAffiliationRorsWithDisplayNamesDf[displayNamesColumnName] = authorAffiliationRorsWithDisplayNamesDf['ror_url'].map(displayNamesDict)

print(f'Rows with no display name: {authorAffiliationRorsWithDisplayNamesDf[displayNamesColumnName].isna().sum()}')

authorAffiliationRorsWithDisplayNamesDf.to_csv(
    os.path.join(desktopPath, 'ror_urls_and_organization_names.csv'),
    index=False, encoding='utf-8-sig')
authorAffiliationRorsWithDisplayNamesDf.head()

# To authorAffiliationsDf, add ror_url_display_name column
To rows in author_affiliations_2026.08.25-2026.09.02.csv that have a ror_url, add a column that shows the display name of the ROR URL from the ROR API, using ror_urls_and_organization_names.csv
Save new dataframe as authorAffiliationsDf3

In [ ]:
# Load the author affiliations and the display name of each ROR URL from the ROR API
authorAffiliationsDf3 = pd.read_csv(
    os.path.join('data', 'processed', 'author_affiliations_2026.08.25-2026.09.02.csv'),
    dtype=str,
    encoding='utf-8-sig')

rorDisplayNamesDf = pd.read_csv(
    os.path.join('data', 'processed', 'ror_urls_and_organization_names.csv'),
    usecols=['ror_url', 'ror_display_names_2026.11.01'],
    dtype=str,
    encoding='utf-8-sig')

print(f'Rows in authorAffiliationsDf3: {len(authorAffiliationsDf3)}')
print(f'Rows in rorDisplayNamesDf: {len(rorDisplayNamesDf)}')
authorAffiliationsDf3.head()

In [ ]:
# Data quality checks before merging: missing values, duplicate ROR URLs and ROR URLs that have no display name
displayNamesColumnName = 'ror_display_names_2026_11_01'

print(f'Rows in rorDisplayNamesDf with no ROR URL: {rorDisplayNamesDf["ror_url"].isna().sum()}')
print(f'Rows in rorDisplayNamesDf with no display name: {rorDisplayNamesDf[displayNamesColumnName].isna().sum()}')
print(f'Duplicate ROR URLs in rorDisplayNamesDf: {rorDisplayNamesDf["ror_url"].duplicated().sum()}')

rowsWithRorCount = authorAffiliationsDf3['author_affiliation_ror'].notna().sum()
print(f'Rows in authorAffiliationsDf3 with a ROR URL: {rowsWithRorCount}')

rorUrlsNotInRorFile = set(authorAffiliationsDf3['author_affiliation_ror'].dropna()) - set(rorDisplayNamesDf['ror_url'])
print(f'ROR URLs in authorAffiliationsDf3 that are not in rorDisplayNamesDf: {len(rorUrlsNotInRorFile)}')

In [ ]:
# Add the ROR display name to rows that have a ROR URL. Rows without a ROR URL get an empty value.
authorAffiliationsDf3 = authorAffiliationsDf3.merge(
    rorDisplayNamesDf,
    how='left',
    left_on='author_affiliation_ror',
    right_on='ror_url').drop(columns='ror_url')

# Rename the column containing the ROR display names
authorAffiliationsDf3 = (
    authorAffiliationsDf3
        .rename(columns={displayNamesColumnName: 'ror_display_names_2026_11_01'}))

# The merge shouldn't add or remove rows
print(f'Rows in authorAffiliationsDf3: {len(authorAffiliationsDf3)}')
print(f'Rows with a ROR URL but no display name: {(authorAffiliationsDf3["author_affiliation_ror"].notna() & authorAffiliationsDf3['ror_display_names_2026_11_01'].isna()).sum()}')

authorAffiliationsDf3[authorAffiliationsDf3['author_affiliation_ror'].notna()].head()

In [ ]:
# Export the new authorAffiliationsDf
authorAffiliationsDf3.to_csv(
    os.path.join(desktopPath, 'author_affiliations_2026.08.25-2026.09.02.csv'),
    index=False, encoding='utf-8-sig')

authorAffiliationsDf3.head()

# Split author_affiliations_2026.08.25-2026.09.02.csv by installation name and installation url
- Get count of datasets in each installation that have one or more author affiliation rors

In [ ]:
# Create dataframe from author_affiliations_2026.08.25-2026.09.02.csv 
authorAffiliationsDf3 = pd.read_csv(
    os.path.join('data', 'processed', 'author_affiliations_2026.08.25-2026.09.02.csv'),
    encoding='utf-8-sig')

authorAffiliationsDf3.head()

In [ ]:
# Data quality checks before counting: 

## Installation names and URLs
print(f'Rows with no installation name: {authorAffiliationsDf3["dataverse_installation_name"].isna().sum()}')
print(f'Rows with no installation URL: {authorAffiliationsDf3["dataverse_installation_url"].isna().sum()}')

## Each installation name should have only one installation URL
urlCountsDf = authorAffiliationsDf3.groupby('dataverse_installation_name')['dataverse_installation_url'].nunique()
print(f'Installation names with more than one installation URL: {(urlCountsDf > 1).sum()}')

## Dataset PIDs 
print(f'Rows with no dataset PID: {authorAffiliationsDf3["dataset_pid"].isna().sum()}')

duplicatePidRowsDf = authorAffiliationsDf3[
    authorAffiliationsDf3.duplicated(subset=['dataset_pid'], keep=False)]
duplicatePidsDf = (
    duplicatePidRowsDf
    .groupby('dataset_pid', as_index=False)
    .agg(
        row_count=('dataset_pid', 'size'),
        installation_count=('dataverse_installation_name', 'nunique'),
        installation_names=('dataverse_installation_name', lambda x: sorted(set(x.dropna()))),
        installation_urls=('dataverse_installation_url', lambda x: sorted(set(x.dropna()))))
    .sort_values('row_count', ascending=False))
print(f'Duplicated dataset_pid values in more than one installation: {(duplicatePidsDf["installation_count"] > 1).sum()}')

In [ ]:
# Get the count of datasets in each installation that have one or more author affiliation RORs.
# Count distinct dataset PIDs, since a dataset has one row per author affiliation
datasetsWithRorsDf = authorAffiliationsDf3[authorAffiliationsDf3['author_affiliation_ror'].notna()]

datasetCountsByInstallationDf = (
    datasetsWithRorsDf
    .groupby(['dataverse_installation_name', 'dataverse_installation_url'], as_index=False)
    .agg(datasets_with_ror_count=('dataset_pid', 'nunique'))
    .sort_values('datasets_with_ror_count', ascending=False)
    .reset_index(drop=True))

# Add the count of all datasets in each installation that are in authorAffiliationsDf3
totalDatasetCountsDf = (
    authorAffiliationsDf3
    .groupby('dataverse_installation_name', as_index=False)
    .agg(datasets_with_author_affiliations_count=('dataset_pid', 'nunique')))
datasetCountsByInstallationDf = datasetCountsByInstallationDf.merge(
    totalDatasetCountsDf, how='left', on='dataverse_installation_name')

print(f'Installations with at least one dataset that has an author affiliation ROR: {len(datasetCountsByInstallationDf)}')
print(f'Datasets with at least one author affiliation ROR: {datasetCountsByInstallationDf["datasets_with_ror_count"].sum()}')

datasetCountsByInstallationDf

In [ ]:
# Create a new dataframe with only the 16 installations that 
# have one or more datasets that have an author affiliation ROR

installationUrlsWithAuthorAffRors = datasetCountsByInstallationDf['dataverse_installation_url'].to_list()

authorAffiliationsOfAffRorInstallationsDf = (authorAffiliationsDf3
    .query(
        'dataverse_installation_url in @installationUrlsWithAuthorAffRors',
    engine='python'
    )
)

authorAffiliationsOfAffRorInstallationsDf.head()

## Version 1

For each installation in authorAffiliationsOfAffRorInstallationsDf:
- Create a directory
- Split authorAffiliationsOfAffRorInstallationsDf by dataverse_installation_url to create a dataframe for each installation
- Get the installation name and URL
- Export each dataframe to the installation's directory
- Get the list of distinct author affiliation names that have ROR URLs and the count of datasets with those author affiliations as names and as ROR URLs
- For each affiliation, search the installation for
  - Datasets with the author affiliation name. Get the count of datasets, stringBasedSearchCount, and the list of distinct dataset PIDs, stringBasedSearchList
  - Datasets with the ROR URL of that author affiliation. Get the count of datasets, rorUrlBasedSearchCount, and the list of distinct dataset PIDs, rorUrlBasedSearchList
  - Get count of distinct datasets in the stringBasedSearchList and the rorUrlBasedSearchList
  - Indicate if there are datasets in the stringBasedSearchList do not include all datasets in the rorUrlBasedSearchList
  - Add columns with:
    - totalDatasetCount (count of distinct datasets in the stringBasedSearchList and the rorUrlBasedSearchList)
    - stringBasedSearchCount
    - rorUrlBasedSearchCount
    - stringBasedSearchReturnsAllDatasets (True/False)

In [ ]:
countOfDistinctAuthorAffiliationNamesByInstallationDict = []
for _, installationDf in authorAffiliationsOfAffRorInstallationsDf.groupby(['dataverse_installation_url']):
    installationDf.reset_index(drop=True)

    # Get the installation name for the installationDf
    installation_name = installationDf['dataverse_installation_name'].iloc[0]
    installation_url = installationDf['dataverse_installation_url'].iloc[0]

    # Make the installation name safe for use in a filename by replacing spaces and special characters with underscores
    safe_installation_name = re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]', '_', installation_name).strip('_'))

    # Count the distinct datasets that have each author affiliation name without a ROR URL
    noRorCountsDf = (
        installationDf[installationDf['author_affiliation_ror'].isna()]
        .groupby('author_affiliation_name', as_index=False)
        .agg(count_of_datasets_with_only_name=('dataset_pid', 'nunique')))

    # Count the distinct datasets that have each author affiliation name with a ROR URL
    # and list the distinct ROR URLs of each name (a name can have more than one ROR URL)
    withRorCountsDf = (
        installationDf[installationDf['author_affiliation_ror'].notna()]
        .groupby('author_affiliation_name', as_index=False)
        .agg(
            ror_urls=('author_affiliation_ror', lambda x: sorted(set(x))),
            count_of_datasets_with_ror=('dataset_pid', 'nunique')))

    # One row per distinct author affiliation name. Names that never appear with (or without) a ROR URL get a count of 0
    affiliationCountsDf = noRorCountsDf.merge(
        withRorCountsDf, how='outer', on='author_affiliation_name')
    affiliationCountsDf[['count_of_datasets_with_only_name', 'count_of_datasets_with_ror']] = (
        affiliationCountsDf[['count_of_datasets_with_only_name', 'count_of_datasets_with_ror']]
        .fillna(0).astype(int))

    # Order the columns
    affiliationCountsDf = affiliationCountsDf[[
        'author_affiliation_name', 'ror_urls',
        'count_of_datasets_with_only_name', 'count_of_datasets_with_ror']]

    # From affiliationCountsDf remove rows where count_of_datasets_with_only_name 
    # and count_of_datasets_with_ror are 0
    affiliationCountsDf = (affiliationCountsDf
        .query(
            'count_of_datasets_with_only_name > 0 and '
            'count_of_datasets_with_ror > 0',
        engine='python'
        ))

    # Sort by count_of_datasets_with_only_name desc and reindex
    affiliationCountsDf = (
        affiliationCountsDf
        .sort_values('count_of_datasets_with_only_name', ascending=False)
        .reset_index(drop=True))

    distinctAffiliationsCount = len(affiliationCountsDf)

    if distinctAffiliationsCount > 0:
        
        # Create directory for the installation
        installationDirPath = os.path.join(desktopPath, 'installations', safe_installation_name)
        os.makedirs(installationDirPath)

        #Export the installationDf dataframe
        fileName =f'{safe_installation_name}.csv'
        filePath = os.path.join(installationDirPath , fileName)
        installationDf.to_csv(filePath, index=False, encoding='utf-8-sig')

        # Export affiliationCountsDf
        fileName = f'author_affiliation_dataset_count.csv'
        filePath = os.path.join(installationDirPath, fileName)
        affiliationCountsDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

        newRow = {
            'installation_name': safe_installation_name,
            'distinct_affiliation_name_count': len(affiliationCountsDf)
        }
        countOfDistinctAuthorAffiliationNamesByInstallationDict.append(newRow)

summaryDf = pd.DataFrame(countOfDistinctAuthorAffiliationNamesByInstallationDict)
summaryDf.head()

## Version 2

In [ ]:
countOfDistinctAuthorAffiliationNamesByInstallationDict = []
for _, installationDf in authorAffiliationsOfAffRorInstallationsDf.groupby(['dataverse_installation_url']):
    installationDf.reset_index(drop=True)

    # Get the installation name for the installationDf
    installation_name = installationDf['dataverse_installation_name'].iloc[0]
    installation_url = installationDf['dataverse_installation_url'].iloc[0]

    # Make the installation name safe for use in a filename by replacing spaces and special characters with underscores
    safe_installation_name = re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]', '_', installation_name).strip('_'))

    # Get list of author affiliations where there's a ROR URL but no affiliation name
    query_expr = """
        (expanded_value_key_present == False and \
        author_affiliation_ror.notnull()) \
        or \
        (expanded_value_key_present == True and \
        author_affiliation_name.isnull())
    """

    # if installation_name == 'DANS Data Station Social Sciences and Humanities':
    # if installation_name == 'Harvard Dataverse':

    rorUrlPrenset_affNameNotPresentDf = (installationDf
        .query(query_expr, engine='python')
        [['author_affiliation_ror', 'ror_display_name_2026_11_01', 'expanded_value_key_present']]
        .drop_duplicates()
        .groupby(['ror_display_name_2026_11_01', 'expanded_value_key_present'])['author_affiliation_ror'].agg(list)
        .reset_index()
        [['ror_display_name_2026_11_01', 'author_affiliation_ror', 'expanded_value_key_present']]
    )

    newRow = {
        'installation_name': safe_installation_name,
        'distinct_affiliation_name_count': len(rorUrlPrenset_affNameNotPresentDf)
    }
    countOfDistinctAuthorAffiliationNamesByInstallationDict.append(newRow)

    if len(rorUrlPrenset_affNameNotPresentDf) > 0:
        
        # Create directory for the installation
        installationDirPath = os.path.join(desktopPath, 'installations_2', safe_installation_name)
        os.makedirs(installationDirPath)

        #Export the installationDf dataframe
        fileName =f'{safe_installation_name}.csv'
        filePath = os.path.join(installationDirPath , fileName)
        installationDf.to_csv(filePath, index=False, encoding='utf-8-sig')

        # Export rorUrlPrenset_affNameNotPresentDf
        fileName = f'rorUrlPrenset_affNameNotPresentDf.csv'
        filePath = os.path.join(installationDirPath, fileName)
        rorUrlPrenset_affNameNotPresentDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

summaryDf = pd.DataFrame(countOfDistinctAuthorAffiliationNamesByInstallationDict)

summaryDf = (summaryDf
    .sort_values('distinct_affiliation_name_count', ascending=False)
    .reset_index(drop=True))

fileName = f'summaryDf.csv'
filePath = os.path.join(desktopPath, 'installations_2', fileName)
summaryDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

summaryDf.head()

# Measure bug's affect on findability of datasets in Dataverse installations

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation